
# 第三章

## 1.状态

dataclasses 和 Pydantic 提供了更强大的数据建模和验证能力。尤其是 Pydantic，它支持运行时数据验证，能确保状态的类型和取值符合期，这对于构建健壮的 AI 智能体系统至关重要。例如，可以使用 Pydantic 定义状态模型，明确指定字段类型（如字符串、整型）并通过校验器约束字段取值范围。

In [1]:
from typing_extensions import TypedDict
from pydantic import BaseModel,field_validator
# 使用 TypedDict 定义状态
class TypedDictState(TypedDict):
  user_input: str
  agent_response: str
  tool_output: str
# 使用 Pydantic 定义状态，并进行数据验证
from pydantic import BaseModel,field_validator
class PydanticState(BaseModel):
  user_input: str
  agent_response: str
  tool_output: str
  mood: str = "neutral" # 默认情绪状态为 neutral
  @field_validator('mood')
  @classmethod
  def validate_mood(cls, value):
    if value not in ["happy", "sad", "neutral"]:
       raise ValueError(" 情绪状态必须是 'happy', 'sad' 或'neutral'")
    return value

## 1.1 私有状态&公共状态


In [ ]:
# 使用多结构体实现私有状态
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, MessagesState
# 定义全局的公共状态结构体 Schema
class OverallState(TypedDict):
    user_input: str
    agent_response: str

# 定义节点的私有状态结构体 Schema
class ToolState(TypedDict):
    api_key:str
    tool_config:dict

 # 定义一个使用私有状态的节点
def tool_node(state:ToolState) -> OverallState:
    # 节点逻辑，例如调用工具 API 并根据 ToolState 中的配置进行操作
    api_client = create_api_client(state['api_key'], state['tool_config'])
    response = api_client.call_api(state['user_input'])
    return {"agent_response":response} # Return the updated publicstate

# Graph 构建代码
builder = StateGraph(OverallState) # 图使用公共状态 OverallState
builder.add_node("tool_node", tool_node) # 添加工具节

In [4]:
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, MessagesState

# 定义完整的内部状态结构体
class InternalState(TypedDict):
    user_query: str
    search_results: list[str]
    llm_response: str
    debug_info: str  # 内部调试信息，不需要对外暴露

# 定义输入结构体 (只包含 user_query)
class InputSchema(TypedDict):
    user_query: str

# 定义输出结构体 (只包含 llm_response)
class OutputSchema(TypedDict):
    llm_response: str

# ✅ 正确写法
graph = StateGraph(
    InternalState,
    input_schema=InputSchema,
    output_schema=OutputSchema
)


In [5]:
# 状态归约器
from typing_extensions import TypedDict, Annotated
from operator import add
from langgraph.graph.message import add_messages
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.messages import BaseMessage
# 定义状态 Schema，并为 message_history 键指定 add_messages Reducer
class ChatState(TypedDict):
    message_history: Annotated[list[BaseMessage], add_messages]
    user_intent: str
    tool_output: str
# 在 Graph 构建时，使用 ChatState 作为状态 Schema
builder = StateGraph(ChatState)

In [6]:
def reducer_extend_unique(left: list[str] | None, right: list[str] | None)->list[str]:
    """
    自定义归约函数，用于合并两个字符串列表，并进行去重
    """
    existing_items = left if left else [] # 如果 left 为 None，则初始化为空列表
    new_items = right if right else [] # 如果 right 为 None，则初始化为空列表
    combined_items = existing_items + new_items
    return list(set(combined_items)) # 使用 set 去重并转换为 list 返回

In [7]:
from typing_extensions import TypedDict, Annotated
class ChatState(TypedDict):
    message_history: Annotated[list[BaseMessage], add_messages]
    user_intent: str
    tool_output: str
    item_list: Annotated[list[str], reducer_extend_unique] # 应用自定义归约函数

In [8]:
from langgraph.graph import MessagesState
class MyChatState(MessagesState):
    """
    自定义的 ChatState， 继承自 MessagesState,
    自动包含 messages 状态键和 add_messages Reducer
    """
    user_intent: str
    tool_output: str
    # 可以添加其他自定义的状态键

In [ ]:
from langchain_core.messages import trim_messages
from langgraph.graph import MessagesState
from langchain_openai import  ChatOpenAI
def llm_node(state: MessagesState):
    message_history = state['messages'] # 直接从 MessageState 中获取消息列表
    trimmed_messages = trim_messages(
        message_history,
        max_tokens=1000,
        strategy="last",
        token_counter=ChatOpenAI(model="gpt-4o"),
        allow_partial=False
    )
    llm_response = llm.invoke(trimmed_messages)
    return {"messages": [llm_response]} # 将 LLM 响应添加到消息历史 (通过 add_messages Reducer)

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import RemoveMessage
def filter_message_node(state: MessagesState): # 节点输入类型为MessageState
    message_history = state['messages'] # 直接从 MessageState 中获取消息列表
    messages_to_remove = []
    for message in message_history:
        if should_remove_message(message): # 假设有函数来判断消息是否需要移除 ( 例如，移除寒暄语 )
            messages_to_remove.append(RemoveMessage(id=message.id))
    return {"messages": messages_to_remove} # 返回RemoveMessage列表，LangGraph 会自动处理

In [ ]:
def my_node(state):
    # 从状态中读取数据
    input_data = state.get("some_key")
    # 执行节点执行逻辑 ( 例如调用 LLM、工具等 )
    output_data = process_data(input_data)
    # 返回新的状态（或状态的更新部分）
    return {"some_key": output_data, "another_key": new_value}

In [9]:
# 包含llm节点的
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from config import OPENAI_API_BASE,OPENAI_API_KEY
# 定义状态结构体
class ChatState(MessagesState): # 使用 ChatState 替代原有的 State，并继承自 MessageState
    user_question: str # 保留 user_question 状态键
    llm_response: str # 保留 llm_response 状态键
# 定义 LLM 节点
def llm_node(state):
    prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
    ])
    model = ChatOpenAI(model="Qwen/Qwen2.5-7B-Instruct",api_key=OPENAI_API_KEY,base_url=OPENAI_API_BASE)
    chain = prompt | model
    response = chain.invoke({"question": state['user_question']}).content
    return {"llm_response": response}
# 构建图
builder = StateGraph(ChatState) # 使用 ChatState 替代原有的 State
builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)
graph = builder.compile()
# 调用图
result = graph.invoke({"user_question": " 你好，LangGraph ！ "})
print(result)

{'messages': [], 'user_question': ' 你好，LangGraph ！ ', 'llm_response': '你好！看来你可能是在以一种特别的方式和我打招呼，但我实际上是Qwen，由阿里云开发的AI助手。我没有听说过LangGraph，不知道你具体指的是什么。如果你有关于LangGraph的问题或是需要任何帮助的地方，请告诉我，我会尽力提供支持！'}


In [12]:
import operator
import sqlite3
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.messages import AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from config import OPENAI_API_BASE, OPENAI_API_KEY


# ==============================
# 1️⃣ 初始化数据库（内存数据库）
# ==============================
db = SQLDatabase.from_uri("sqlite:///:memory:")

# 建表
db.run("""
CREATE TABLE IF NOT EXISTS Artist (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    genre TEXT,
    country TEXT
);
""")

# 插入一些示例数据
db.run("""
INSERT INTO Artist (name, genre, country) VALUES
('周杰伦', '流行', '中国'),
('Taylor Swift', 'Pop', 'USA'),
('BTS', 'K-pop', '韩国'),
('Adele', 'Soul', 'UK'),
('陈奕迅', '流行', '中国香港'),
('Ed Sheeran', 'Pop', 'UK'),
('Blackpink', 'K-pop', '韩国'),
('The Weeknd', 'R&B', 'Canada'),
('张学友', '流行', '中国香港'),
('五月天', '摇滚', '中国台湾');
""")


# ==============================
# 2️⃣ 定义模型
# ==============================
model = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct",
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE
)


# ==============================
# 3️⃣ 定义状态与节点
# ==============================
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]


def query_database(state):
    try:
        query_result = db.run("SELECT * FROM Artist LIMIT 10;")
        return {"messages": [AIMessage(content=query_result)]}
    except Exception as e:
        return {"messages": [AIMessage(content=f"❌ 查询出错: {e}")]}


def call_model(state):
    response = model.invoke(state["messages"])
    return {"messages": [response]}


# ==============================
# 4️⃣ 定义图与重试策略
# ==============================
builder = StateGraph(AgentState)

builder.add_node(
    "query_database",
    query_database,
    retry=RetryPolicy(retry_on=sqlite3.OperationalError)
)

builder.add_node(
    "model",
    call_model,
    retry=RetryPolicy(max_attempts=5)
)

builder.add_edge(START, "model")
builder.add_edge("model", "query_database")
builder.add_edge("query_database", END)

graph = builder.compile()
# 调用图
from langchain_core.messages import HumanMessage

# 定义初始输入消息
initial_state = {
    "messages": [HumanMessage(content="请查询数据库中的歌手信息")]
}

# 调用 graph
result = graph.invoke(initial_state)

# 查看输出结果
print(result)

{'messages': [HumanMessage(content='请查询数据库中的歌手信息', additional_kwargs={}, response_metadata={}), AIMessage(content='要查询数据库中的歌手信息，首先需要知道数据库的具体信息，例如数据库的名称、连接数据库所需的用户名和密码，以及数据库中表示歌手信息的表的名称和列。假设我们已经有了这些信息，下面是一个使用SQL查询的例子。同样，假设数据库使用的是MySQL，并且歌手信息存储于一个名为`musician`的表中，其中包含`id`（歌手ID）、`name`（歌手姓名）、`genre`（音乐风格）等列。\n\n### SQL 查询示例\n```sql\nSELECT id, name, genre, birth_date, nationality FROM musician;\n```\n这条查询将返回歌手表格中的每个歌手的名字、所属音乐流派、生日和国籍。\n\n### 如何连接到数据库\n如果尚未连接到数据库，可以使用对应的数据库连接工具或库（例如Python中的pyodbc, MySQLdb, SQLAlchemy等）来完成初始化。下面是一个使用Python和`mysql-connector-python`库连接数据库的简单示例：\n\n```python\nimport mysql.connector\n\n# 连接到MySQL数据库\ntry:\n    connection = mysql.connector.connect(\n        host=\'hostname\',  # 数据库主机,例如localhost\n        user=\'username\',  # 数据库用户名\n        password=\'password\',  # 数据库密码\n        database=\'database_name\'  # 数据库名称\n    )\n    if connection.is_connected():\n        db_info = connection.get_server_info()\n        print("成功连接到数据库 version: ", db_info)\n        \n

In [17]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
class State(TypedDict):
    topic: str
    joke: str
def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}
def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile()
)
for chunk in graph.stream({"topic": "ice cream"},stream_mode="values",):
    print(chunk)



{'topic': 'ice cream'}
{'topic': 'ice cream and cats'}
{'topic': 'ice cream and cats', 'joke': 'This is a joke about ice cream and cats'}


In [18]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from config import OPENAI_API_BASE,OPENAI_API_KEY
class State(TypedDict):
    topic: str
    joke: str
llm = ChatOpenAI(model="Qwen/Qwen2.5-7B-Instruct",api_key=OPENAI_API_KEY,base_url=OPENAI_API_BASE)
def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}
def generate_joke(state: State):
    llm_response = llm.invoke([{"role": "user", "content": f"Generate a joke about {state['topic']}"}])
    return {"joke": llm_response.content}
graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile(checkpointer=MemorySaver())
)
config = {"configurable": {"thread_id": "my_thread_1"}}
for chunk in graph.stream({"topic": "ice cream"},config=config,stream_mode="updates",):
    print(chunk)
    print(graph.get_state(config).values)

state_history = list(graph.get_state_history(config))
for snapshot in state_history:
    print(f" 存档点 ID: {snapshot.config['configurable']['checkpoint_id']}")
    print(f" 步骤元数据 : {snapshot.metadata}")
    print(f" 父图状态值 : {snapshot.values}")
    print(f" 下一个节点 : {snapshot.next}")
    print("=" * 20)

{'refine_topic': {'topic': 'ice cream and cats'}}
{'topic': 'ice cream and cats'}
{'generate_joke': {'joke': 'Why did the cat choose chocolate ice cream over vanilla?\n\nBecause chocolate has whiskers!'}}
{'topic': 'ice cream and cats', 'joke': 'Why did the cat choose chocolate ice cream over vanilla?\n\nBecause chocolate has whiskers!'}
 存档点 ID: 1f0aa6a8-114c-628b-8002-92a8a39cf1a1
 步骤元数据 : {'source': 'loop', 'step': 2, 'parents': {}}
 父图状态值 : {'topic': 'ice cream and cats', 'joke': 'Why did the cat choose chocolate ice cream over vanilla?\n\nBecause chocolate has whiskers!'}
 下一个节点 : ()
 存档点 ID: 1f0aa6a8-04ab-69e6-8001-429468eeefa8
 步骤元数据 : {'source': 'loop', 'step': 1, 'parents': {}}
 父图状态值 : {'topic': 'ice cream and cats'}
 下一个节点 : ('generate_joke',)
 存档点 ID: 1f0aa6a8-04a9-6e50-8000-ea522b152901
 步骤元数据 : {'source': 'loop', 'step': 0, 'parents': {}}
 父图状态值 : {'topic': 'ice cream'}
 下一个节点 : ('refine_topic',)
 存档点 ID: 1f0aa6a8-036b-6375-bfff-629cb83fd5f2
 步骤元数据 : {'source': 'input', '

In [19]:
# 假设来自先前示例的 state_history
checkpoint_to_replay = state_history[2] # 让我们从历史记录中的第 3 个存档点重放
replay_config = checkpoint_to_replay.config
for event in graph.stream(None, replay_config, stream_mode="values"):
    print(event)

{'topic': 'ice cream'}
{'topic': 'ice cream and cats'}
{'topic': 'ice cream and cats', 'joke': 'Why did the cat refuse to eat ice cream?\n\nBecause it was purr-fectly fine on its own!'}
